# v3.1 add-on: label reliability and mean reversion

Run this after the v3 notebook. It reuses the v3 code (data loading, snapshots, community detection, tracking, triad features, purged rolling-origin evaluation) **unchanged** and adds only the diagnostics below. These were specified after reading the v3 results, so in the manuscript they are **post-hoc, exploratory analyses**. Disclose them as such.

**Why.** In v3, the *persistence* baseline (predict that the last size change continues) scored macro-F1 0.20–0.25 in all four modelled networks. That is *below* the stratified-random level of about 0.33, so size changes tend to **reverse**. In the same run, an RF on size alone matched the full model. One explanation is that much of the Growing/Shrinking signal is **community-detection noise**: if Louvain happens to give a community an inflated size at t, the measured change t−1→t is too high and t→t+k too low. That is regression to the mean, and it is predictable from size features alone.

| ID | Question | Output |
|---|---|---|
| D1 | How strongly does the next size change reverse the last one? | cluster-robust slope and Spearman ρ of `rel_change_next` on `rel_d_size` |
| D2 | How reliable are the labels? Re-detect communities with 5 other Louvain seeds, match each community by Jaccard, and recompute its label | agreement and Cohen's κ against the seed-42 labels; relative s.d. of community size across seeds |
| D3 | Does seed-averaging the size (a less noisy measurement) remove the reversal? | D1 recomputed on seed-averaged sizes |
| D4 | Is a two-feature mean-reversion model as good as the full macrostate RF? | multinomial logit on (`rel_d_size`, `log_size`) vs combined RF, same purged rolling-origin folds, paired cluster bootstrap |

Datasets and windows are the ones v3 selected (edit `DIAG_WINDOWS` if your v3 run chose differently).

In [ ]:
# ======================= USER CONFIGURATION =======================
SYNTHETIC_SMOKE_TEST = False   # True -> run end-to-end on two small synthetic networks (no downloads)
RUN_ROBUSTNESS       = True    # Jaccard / label-threshold / seed / algorithm sweeps (slowest part)
RUN_LSTM             = True    # RF vs LSTM comparison (needs torch; preinstalled in Colab)
INCLUDE_WIKI_TALK    = False   # adds the large wiki-talk-temporal network (~7.8M events)
INCLUDE_STACKEXCHANGE = True   # adds sx-askubuntu and sx-superuser (two more Q&A replications)
SAVE_TO_DRIVE        = True   # copy results to Google Drive at the end

CONFIG = dict(
    SEED=42,
    MIN_COMM_SIZE=4,          # smallest community in which internal triads exist
    JACCARD_THETA=0.15,       # community-tracking threshold
    LABEL_THRESHOLD=0.10,     # +/-10% size change -> Growing / Shrinking
    DEATH_AS_SHRINK=False,    # primary: a community that disappears at t+k is unlabelled (sensitivity: True)
    CD_ALGO='louvain',        # 'louvain' | 'leiden' | 'lpa'
    CD_SEED=42,
    # --- rolling-origin evaluation ---
    EVAL_START=0.50,          # first evaluation cut at 50% of the snapshots
    N_EVAL_FOLDS=4,           # cuts spread evenly over the second half; predictions pooled
    N_CV_FOLDS=5,             # grouped CV inside the first fold's training data (tuning only)
    N_RS_ITER=25,             # randomized-search iterations per feature set
    N_RF_SEEDS=5,             # seed ensemble for every random forest
    N_BOOT=1000,              # cluster-bootstrap replicates
    N_ABLATION_SEEDS=3,
    HMM_STATES=[2, 3, 4, 5, 6, 7, 8],
    HMM_RESTARTS=5,
    LSTM_SEEDS=5,
    ROBUSTNESS_DATASETS='all',   # 'all' or a list of dataset keys
    # --- window selection (outcome-blind; Section 10b) ---
    # (m, k): window width = m x stride, label compares t with t+k. Primary candidates are NON-overlapping (k = m),
    # tried in this order; the first that passes the yield rule is used.
    WINDOW_CANDIDATES=[(1, 1), (2, 2), (3, 3), (4, 4), (6, 6)],
    ROBUSTNESS_WINDOWS=[(1, 1), (2, 2), (3, 3), (4, 4), (6, 6), (2, 1), (3, 1), (4, 1)],   # k < m: overlapping, sweep only
    MIN_INSTANCES=200,        # yield rule
    MIN_PER_CLASS=25,
    MIN_COMMUNITIES=10,       # >= grouped-CV folds; cluster-robust SEs need several clusters
    MIN_TEST_INSTANCES=60,    # labelled instances in the evaluation period (t >= first cut)
    MIN_TEST_PER_CLASS=10,
    MIN_TRAIN_INSTANCES=60,   # purged training instances before the first cut
    JACCARD_SWEEP=[0.05, 0.10, 0.15, 0.20, 0.25, 0.30],
    LABEL_SWEEP=[0.05, 0.10, 0.15, 0.20],
    CD_SEED_SWEEP=[0, 1, 2, 3, 4],
    CD_ALGO_SWEEP=['louvain', 'leiden', 'lpa'],
)

DATASETS = {
    'email-Eu-core': dict(url='https://snap.stanford.edu/data/email-Eu-core-temporal.txt.gz',
                          sep='ws', cols=(0, 1, 2), stride_days=14, domain='institutional e-mail'),
    'CollegeMsg':    dict(url='https://snap.stanford.edu/data/CollegeMsg.txt.gz',
                          sep='ws', cols=(0, 1, 2), stride_days=7, domain='online social messaging'),
    'bitcoin-otc':   dict(url='https://snap.stanford.edu/data/soc-sign-bitcoinotc.csv.gz',
                          sep=',', cols=(0, 1, 3), stride_days=30, domain='trust network'),
    'bitcoin-alpha': dict(url='https://snap.stanford.edu/data/soc-sign-bitcoinalpha.csv.gz',
                          sep=',', cols=(0, 1, 3), stride_days=30, domain='trust network'),
    'mathoverflow':  dict(url='https://snap.stanford.edu/data/sx-mathoverflow.txt.gz',
                          sep='ws', cols=(0, 1, 2), stride_days=30, domain='Q&A interactions'),
    # subreddit -> subreddit hyperlinks (body + title files); nodes (subreddits) persist for years
    'reddit-hyperlinks': dict(urls=['https://snap.stanford.edu/data/soc-redditHyperlinks-body.tsv',
                                    'https://snap.stanford.edu/data/soc-redditHyperlinks-title.tsv'],
                          sep='\t', cols=(0, 1, 3), ts_format='datetime', stride_days=30,
                          domain='online community hyperlinks'),
}
if INCLUDE_STACKEXCHANGE:
    DATASETS['askubuntu'] = dict(url='https://snap.stanford.edu/data/sx-askubuntu.txt.gz',
                                 sep='ws', cols=(0, 1, 2), stride_days=30, domain='Q&A interactions')
    DATASETS['superuser'] = dict(url='https://snap.stanford.edu/data/sx-superuser.txt.gz',
                                 sep='ws', cols=(0, 1, 2), stride_days=30, domain='Q&A interactions')
if INCLUDE_WIKI_TALK:
    DATASETS['wiki-talk'] = dict(url='https://snap.stanford.edu/data/wiki-talk-temporal.txt.gz',
                                 sep='ws', cols=(0, 1, 2), stride_days=30, domain='collaborative talk pages')

if SYNTHETIC_SMOKE_TEST:
    DATASETS = {
        'synthA': dict(synthetic=dict(n_nodes=260, n_comms=8, days=420, events_per_day=160, seed=1),
                       stride_days=14, domain='synthetic A'),
        'synthB': dict(synthetic=dict(n_nodes=200, n_comms=6, days=300, events_per_day=140, seed=2),
                       stride_days=10, domain='synthetic B'),
    }
    CONFIG.update(N_RS_ITER=4, N_BOOT=100, N_ABLATION_SEEDS=2, N_RF_SEEDS=2, HMM_RESTARTS=2, LSTM_SEEDS=2,
                  HMM_STATES=[2, 3], N_EVAL_FOLDS=2, ROBUSTNESS_DATASETS=['synthA'], JACCARD_SWEEP=[0.10, 0.20],
                  LABEL_SWEEP=[0.10, 0.20], CD_SEED_SWEEP=[0, 1], CD_ALGO_SWEEP=['louvain', 'lpa'],
                  WINDOW_CANDIDATES=[(1, 1), (2, 2)], ROBUSTNESS_WINDOWS=[(1, 1), (2, 1)],
                  MIN_INSTANCES=100, MIN_PER_CLASS=5, MIN_COMMUNITIES=3,
                  MIN_TEST_INSTANCES=20, MIN_TEST_PER_CLASS=2, MIN_TRAIN_INSTANCES=20)

import os
import sys
IN_COLAB = 'google.colab' in sys.modules
BASE_DIR    = '/content' if IN_COLAB else os.getcwd()
DATA_DIR    = os.path.join(BASE_DIR, 'snap_data')
RESULTS_DIR = os.path.join(BASE_DIR, 'results_diagnostics')
PIPELINE_VERSION = '3.1-diagnostics (label reliability and mean reversion; reuses v3 code unchanged)'
import shutil
if os.path.isdir(RESULTS_DIR):
    shutil.rmtree(RESULTS_DIR)          # never mix files from an earlier run (e.g. a smoke test) into this one
os.makedirs(DATA_DIR, exist_ok=True); os.makedirs(RESULTS_DIR, exist_ok=True)
print('Pipeline version:', PIPELINE_VERSION)
print('Datasets:', list(DATASETS)); print('Results ->', RESULTS_DIR, '(cleared)')

In [ ]:
import sys, subprocess, importlib.util
def _ensure(pkgs):
    missing = [pip for pip, mod in pkgs if importlib.util.find_spec(mod) is None]
    if missing:
        print('Installing:', missing)
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing])
_ensure([('hmmlearn', 'hmmlearn'), ('shap', 'shap'), ('lifelines', 'lifelines'),
         ('statsmodels', 'statsmodels'), ('python-igraph', 'igraph'), ('leidenalg', 'leidenalg')])

In [ ]:
import os, gzip, json, math, time, shutil, pickle, random, warnings, urllib.request
from collections import Counter, defaultdict
import numpy as np, pandas as pd, networkx as nx
import matplotlib.pyplot as plt
from scipy import stats
from scipy.special import logsumexp
import statsmodels.api as sm
from statsmodels.stats.multicomp import pairwise_tukeyhsd
from statsmodels.stats.multitest import multipletests
from statsmodels.stats.contingency_tables import mcnemar
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import GroupKFold, RandomizedSearchCV
from sklearn.metrics import (accuracy_score, f1_score, balanced_accuracy_score,
                             classification_report, confusion_matrix)
from sklearn.inspection import permutation_importance
import shap
from hmmlearn.hmm import GaussianHMM
from lifelines import KaplanMeierFitter, CoxPHFitter
from lifelines.statistics import logrank_test
try:
    import torch, torch.nn as nn
    HAS_TORCH = True
except ImportError:
    HAS_TORCH = False
try:
    import igraph as ig, leidenalg as la
    HAS_LEIDEN = True
except ImportError:
    HAS_LEIDEN = False

warnings.filterwarnings('ignore')
pd.set_option('display.width', 160); pd.set_option('display.max_columns', 40)
SEED = CONFIG['SEED']; random.seed(SEED); np.random.seed(SEED)
CLASSES = ['Growing', 'Stable', 'Shrinking']
# Okabe-Ito colour-blind-safe colours, fixed per class (never cycled)
CLASS_COLORS = {'Growing': '#009E73', 'Stable': '#7f7f7f', 'Shrinking': '#D55E00'}
plt.rcParams.update({'figure.dpi': 110, 'axes.spines.top': False, 'axes.spines.right': False,
                     'axes.grid': True, 'grid.alpha': 0.25, 'font.size': 9})
print('python', sys.version.split()[0], '| networkx', nx.__version__, '| shap', shap.__version__,
      '| torch' if HAS_TORCH else '| (no torch)', torch.__version__ if HAS_TORCH else '',
      '| leiden' if HAS_LEIDEN else '| (no leiden)')

In [ ]:
def download(url):
    path = os.path.join(DATA_DIR, os.path.basename(url))
    if not os.path.exists(path):
        print('  downloading', url)
        req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0 (research pipeline)'})
        with urllib.request.urlopen(req, timeout=300) as r, open(path + '.part', 'wb') as f:
            shutil.copyfileobj(r, f)
        os.replace(path + '.part', path)
    return path

def synthetic_events(n_nodes, n_comms, days, events_per_day, seed):
    # Planted communities whose membership drifts (nodes migrate, communities grow and shrink).
    rng = np.random.default_rng(seed)
    memb = rng.integers(0, n_comms, n_nodes)
    pull = rng.normal(0, 1, n_comms)
    rows = []
    for d in range(days):
        if d % 7 == 0:                        # weekly migration biased by a slowly drifting 'pull'
            pull = 0.9 * pull + rng.normal(0, 0.5, n_comms)
            p = np.exp(pull) / np.exp(pull).sum()
            movers = rng.random(n_nodes) < 0.04
            memb[movers] = rng.choice(n_comms, movers.sum(), p=p)
        for _ in range(events_per_day):
            u = rng.integers(n_nodes)
            same = np.flatnonzero(memb == memb[u])
            v = rng.choice(same) if (rng.random() < 0.85 and len(same) > 1) else rng.integers(n_nodes)
            if u != v:
                rows.append((u, v, d * 86400 + rng.integers(86400)))
                if rng.random() < 0.3: rows.append((v, u, d * 86400 + rng.integers(86400)))
    return pd.DataFrame(rows, columns=['src', 'dst', 'ts'])

def load_events(name, spec):
    if 'synthetic' in spec:
        ev = synthetic_events(**spec['synthetic'])
    else:
        sep = r'\s+' if spec['sep'] == 'ws' else spec['sep']
        frames = []
        for url in spec.get('urls', [spec.get('url')]):
            f = pd.read_csv(download(url), sep=sep, header=None, comment=None if spec.get('ts_format') else '#',
                            usecols=list(spec['cols']), dtype=str)
            f.columns = ['src', 'dst', 'ts']; frames.append(f)
        ev = pd.concat(frames, ignore_index=True)
        if spec.get('ts_format') == 'datetime':
            ts = pd.to_datetime(ev.ts, errors='coerce')            # header rows -> NaT, dropped below
            ev = ev.assign(ts=(ts - pd.Timestamp('1970-01-01')) // pd.Timedelta('1s')).dropna()
        else:
            ev = ev.assign(ts=pd.to_numeric(ev.ts, errors='coerce'))
        ev = ev.dropna()
        if not (pd.to_numeric(ev.src, errors='coerce').notna().all() and pd.to_numeric(ev.dst, errors='coerce').notna().all()):
            codes, _ = pd.factorize(pd.concat([ev.src, ev.dst], ignore_index=True))   # string node ids -> ints
            ev = ev.assign(src=codes[:len(ev)], dst=codes[len(ev):])
    ev = ev.dropna()
    ev = ev[ev.src != ev.dst].astype({'src': 'int64', 'dst': 'int64', 'ts': 'int64'})
    return ev.sort_values('ts').reset_index(drop=True)

def build_snapshots(ev, width_days, stride_days):
    # Window k covers [t0 + k*stride, t0 + k*stride + width). Only complete windows are kept.
    w, s = int(width_days * 86400), int(stride_days * 86400)
    t0, T = ev.ts.min(), ev.ts.max() - ev.ts.min()
    n_win = max(1, int((T - w) // s) + 1)
    ts = ev.ts.values; snaps = []
    for k in range(n_win):
        lo, hi = np.searchsorted(ts, [t0 + k * s, t0 + k * s + w], side='left')
        grp = ev.iloc[lo:hi]
        dcount = grp.groupby(['src', 'dst']).size()
        directed = set(dcount.index)
        und = Counter()
        for (u, v), c in dcount.items():
            und[(u, v) if u < v else (v, u)] += c
        G = nx.Graph(); G.add_weighted_edges_from((u, v, c) for (u, v), c in und.items())
        snaps.append(dict(G=G, directed=directed, n_events=len(grp)))
    return snaps

In [ ]:
def detect_communities(G, algo='louvain', seed=42, min_size=4):
    if G.number_of_edges() == 0:
        return []
    if algo == 'louvain':
        comms = nx.community.louvain_communities(G, weight='weight', seed=seed)
    elif algo == 'leiden':
        if not HAS_LEIDEN: raise RuntimeError('leidenalg not installed')
        g = ig.Graph.from_networkx(G)
        part = la.find_partition(g, la.ModularityVertexPartition, weights='weight', seed=seed)
        names = g.vs['_nx_name']
        comms = [{names[i] for i in c} for c in part]
    elif algo == 'lpa':
        comms = nx.community.asyn_lpa_communities(G, weight='weight', seed=seed)
    else:
        raise ValueError(algo)
    return sorted([frozenset(c) for c in comms if len(c) >= min_size], key=lambda c: -len(c))

def track_communities(parts, theta):
    # Returns ids[t][j] = persistent community id of parts[t][j].
    ids, next_id, prev, prev_ids = [], 0, None, None
    for comms in parts:
        cur = [None] * len(comms)
        if prev:
            node2prev = {u: i for i, c in enumerate(prev) for u in c}
            cands = []
            for j, c in enumerate(comms):
                ov = Counter(node2prev[u] for u in c if u in node2prev)
                for i, o in ov.items():
                    jac = o / (len(prev[i]) + len(c) - o)
                    if jac >= theta: cands.append((jac, i, j))
            used = set()
            for jac, i, j in sorted(cands, reverse=True):
                if i in used or cur[j] is not None: continue
                cur[j] = prev_ids[i]; used.add(i)
        for j in range(len(comms)):
            if cur[j] is None: cur[j] = next_id; next_id += 1
        ids.append(cur); prev, prev_ids = comms, cur
    return ids

In [ ]:
MACRO_K = 5

def community_stats(G, directed, C):
    C = set(C); H = G.subgraph(C)
    n, m = len(C), H.number_of_edges()
    tri = sum(nx.triangles(H).values()) // 3
    wedges = sum(d * (d - 1) // 2 for _, d in H.degree())
    s1, s2 = tri, wedges - 3 * tri                       # node-triples: closed / open
    out = {u: {v for v in G[u] if v not in C} for u in C}
    kx = Counter(x for u in C for x in out[u])
    pairs_x = sum(k * (k - 1) // 2 for k in kx.values())
    s3 = s5 = 0
    for a, b in H.edges():
        c = len(out[a] & out[b]); s3 += c
        s5 += len(out[a]) + len(out[b]) - 2 * c
    s4 = pairs_x - s3
    counts = np.array([s1, s2, s3, s4, s5], dtype=float)
    tot = counts.sum()
    shares = counts / tot if tot > 0 else np.zeros(MACRO_K)
    nz = shares[shares > 0]
    cut = sum(len(o) for o in out.values())
    vol_c = sum(G.degree(u) for u in C); vol_r = 2 * G.number_of_edges() - vol_c
    recip = (sum(1 for a, b in H.edges() if (a, b) in directed and (b, a) in directed) / m) if m else 0.0
    return dict(size=n, density=2 * m / (n * (n - 1)), conductance=cut / max(1, min(vol_c, vol_r)),
                reciprocity=recip, counts=counts, shares=shares,
                entropy=float(-(nz * np.log(nz)).sum()),
                closure_rate=s1 / (s1 + s2) if (s1 + s2) > 0 else 0.0,
                boundary_ratio=(s3 + s4 + s5) / tot if tot > 0 else 0.0,
                Cb=s3, Ob=s4 + s5)

STRUCT = ['log_size', 'density', 'conductance', 'reciprocity', 'rel_d_size']
SHARES = [f'S{k+1}_share' for k in range(MACRO_K)]
MACRO  = SHARES + ['entropy', 'd_entropy', 'closure_rate', 'd_closure_rate',
                   'boundary_ratio', 'd_boundary_ratio', 'BCR']
COMBINED = STRUCT + MACRO
SIZE_ONLY = ['log_size', 'rel_d_size']
FEATURE_SETS = {'size only': SIZE_ONLY, 'structural': STRUCT, 'macrostate': MACRO, 'combined': COMBINED}
ABLATION_GROUPS = {
    'size (log_size, rel_d_size)': ['log_size', 'rel_d_size'],
    'density': ['density'],
    'conductance + reciprocity': ['conductance', 'reciprocity'],
    'internal closure (S1, S2, closure rate, d_closure)': ['S1_share', 'S2_share', 'closure_rate', 'd_closure_rate'],
    'boundary (S3-S5, ratio, d_ratio, BCR)': ['S3_share', 'S4_share', 'S5_share', 'boundary_ratio', 'd_boundary_ratio', 'BCR'],
    'entropy (H, dH)': ['entropy', 'd_entropy'],
    'ALL macrostate features': MACRO,
    'ALL structural features': STRUCT,
}

def snapshot_stats(snaps, parts):
    return [[community_stats(s['G'], s['directed'], c) for c in comms] for s, comms in zip(snaps, parts)]

def label_from_change(r, thr):
    return 'Growing' if r > thr else ('Shrinking' if r < -thr else 'Stable')

def build_table(stats, ids, n_snap, label_thr, death_as_shrink=False, h=1):
    # One row per (community id, snapshot). Features need t-1; the label compares t with t+h
    # (h = label step k; h = m means the label window shares no events with window t).
    rec = {(cid, t): s for t, (ss, ii) in enumerate(zip(stats, ids)) for s, cid in zip(ss, ii)}
    rows = []
    for (cid, t), s in rec.items():
        row = dict(cid=cid, t=t, size=s['size'], log_size=np.log(s['size']), density=s['density'],
                   conductance=s['conductance'], reciprocity=s['reciprocity'], entropy=s['entropy'],
                   closure_rate=s['closure_rate'], boundary_ratio=s['boundary_ratio'],
                   n_triads=s['counts'].sum(), Cb=s['Cb'], Ob=s['Ob'])
        row.update({k: v for k, v in zip(SHARES, s['shares'])})
        p = rec.get((cid, t - 1))
        if p is not None:
            row.update(rel_d_size=(s['size'] - p['size']) / p['size'],
                       d_entropy=s['entropy'] - p['entropy'],
                       d_closure_rate=s['closure_rate'] - p['closure_rate'],
                       d_boundary_ratio=s['boundary_ratio'] - p['boundary_ratio'],
                       BCR=(s['Cb'] - p['Cb']) / max(p['Ob'], 1.0))
        ph = rec.get((cid, t - h))
        if ph is not None: row['rel_d_size_h'] = (s['size'] - ph['size']) / ph['size']
        nx_ = rec.get((cid, t + h))
        n_next = nx_['size'] if nx_ is not None else (0 if (death_as_shrink and t + h < n_snap) else None)
        if n_next is not None:
            r = (n_next - s['size']) / s['size']
            row.update(rel_change_next=r, vol_next=abs(r), label=label_from_change(r, label_thr))
        row['born'] = (cid, t - 1) not in rec
        row['dies_next'] = (cid, t + 1) not in rec and t + 1 < n_snap
        rows.append(row)
    tab = pd.DataFrame(rows).sort_values(['cid', 't']).reset_index(drop=True)
    tab['age'] = tab.t - tab.groupby('cid').t.transform('min')   # snapshots since first tracked
    for c in ['rel_d_size', 'rel_d_size_h', 'd_entropy', 'd_closure_rate', 'd_boundary_ratio', 'BCR',
              'rel_change_next', 'vol_next', 'label']:
        if c not in tab: tab[c] = np.nan
    return tab

def instances_from(tab):
    return tab.dropna(subset=COMBINED + ['label']).reset_index(drop=True)

In [ ]:
RF_SPACE = dict(n_estimators=[200, 400, 600], max_depth=[None, 4, 6, 8, 12],
                min_samples_leaf=[1, 2, 4, 8], max_features=['sqrt', 0.5, None],
                class_weight=[None, 'balanced', 'balanced_subsample'])

def eval_cuts(n_snap):
    s, F = CONFIG['EVAL_START'], CONFIG['N_EVAL_FOLDS']
    cuts = sorted(set(int(np.floor(n_snap * (s + i * (1 - s) / F))) for i in range(F)))
    return cuts + [n_snap]

def purged_train(df, cut, m, k):
    # every event a training row touches (feature windows <= t, label window t+k spanning strides
    # t+k .. t+k+m-1) lies strictly before stride `cut`
    return df[df.t + k + m <= cut]

def make_folds(df, n_snap, m, k):
    c, out = eval_cuts(n_snap), []
    for i in range(len(c) - 1):
        tr = purged_train(df, c[i], m, k); te = df[(df.t >= c[i]) & (df.t < c[i + 1])]
        if len(te) and len(tr) and tr.label.nunique() >= 2 and tr.cid.nunique() >= 2:
            out.append((i, c[i], tr, te))
    return out

def temporal_split(df, n_snap, m, k):
    # single purged split at the first evaluation cut (used by transfer and RF-vs-LSTM)
    c0 = eval_cuts(n_snap)[0]
    return purged_train(df, c0, m, k), df[df.t >= c0], c0

def n_group_folds(groups, k):
    return max(2, min(k, len(np.unique(groups))))

def tune_rf(X, y, groups, seed=SEED):
    cv = GroupKFold(n_splits=n_group_folds(groups, CONFIG['N_CV_FOLDS']))
    rs = RandomizedSearchCV(RandomForestClassifier(random_state=seed, n_jobs=-1), RF_SPACE,
                            n_iter=CONFIG['N_RS_ITER'], scoring='f1_macro', cv=cv,
                            random_state=seed, n_jobs=1, refit=True)
    rs.fit(X, y, groups=groups)
    return rs.best_estimator_, rs.best_params_, rs.best_score_

def macro_f1(y, yhat):
    return f1_score(y, yhat, labels=CLASSES, average='macro', zero_division=0)

def metrics(y, yhat):
    return dict(accuracy=accuracy_score(y, yhat), macro_f1=macro_f1(y, yhat),
                balanced_acc=balanced_accuracy_score(y, yhat))

def _boot_index(groups, n_boot, seed):
    rng = np.random.default_rng(seed); groups = np.asarray(groups)
    ug = np.unique(groups); idx = {g: np.flatnonzero(groups == g) for g in ug}
    for _ in range(n_boot):
        yield np.concatenate([idx[g] for g in rng.choice(ug, len(ug), replace=True)])

def cluster_bootstrap_ci(y, yhat, groups, n_boot=None, seed=SEED):
    y, yhat = np.asarray(y), np.asarray(yhat); out = defaultdict(list)
    for sel in _boot_index(groups, n_boot or CONFIG['N_BOOT'], seed):
        for k, v in metrics(y[sel], yhat[sel]).items(): out[k].append(v)
    return {k: (np.percentile(v, 2.5), np.percentile(v, 97.5)) for k, v in out.items()}

def paired_cluster_boot(y, pa, pb, groups, n_boot=None, seed=SEED):
    # macro-F1(a) - macro-F1(b) on the same instances; communities resampled; two-sided bootstrap p
    y, pa, pb = map(np.asarray, (y, pa, pb))
    obs = macro_f1(y, pa) - macro_f1(y, pb)
    d = np.array([macro_f1(y[s], pa[s]) - macro_f1(y[s], pb[s]) for s in _boot_index(groups, n_boot or CONFIG['N_BOOT'], seed)])
    p = min(1.0, 2 * min((d <= 0).mean(), (d >= 0).mean()))
    return obs, (np.percentile(d, 2.5), np.percentile(d, 97.5)), p

def rf_seed_ensemble(tr, te, cols, params, seeds):
    # average class probabilities over seeds; returns ensemble labels, per-seed labels, first fitted model
    P = np.zeros((len(te), len(CLASSES))); per_seed, first = [], None
    for s in seeds:
        est = RandomForestClassifier(random_state=s, n_jobs=-1, **params).fit(tr[cols], tr.label)
        first = est if first is None else first
        pr, full = est.predict_proba(te[cols]), np.zeros((len(te), len(CLASSES)))
        for j, c in enumerate(est.classes_): full[:, CLASSES.index(c)] = pr[:, j]
        P += full; per_seed.append(np.array(CLASSES)[full.argmax(1)])
    return np.array(CLASSES)[P.argmax(1)], per_seed, first

def persistence_baseline(df, thr):
    # last observed change over the same horizon as the label (t-h -> t); falls back to t-1 -> t
    r = df.rel_d_size_h.fillna(df.rel_d_size)
    return r.apply(lambda v: label_from_change(v, thr)).values

def fmt_ci(v, ci): return f'{v:.3f} [{ci[0]:.3f}, {ci[1]:.3f}]'

In [ ]:
def zscore(df, cols):
    sd = df[cols].std(ddof=0).replace(0, 1)
    return (df[cols] - df[cols].mean()) / sd

def cluster_fit(df, y, xs, kind='logit', group='cid'):
    d = df[[y] + xs + [group]].replace([np.inf, -np.inf], np.nan).dropna()
    if len(d) < len(xs) + 10 or d[y].nunique() < 2:
        return None, None
    X = sm.add_constant(zscore(d, xs)); g = pd.factorize(d[group])[0]
    model = sm.Logit(d[y].astype(float), X) if kind == 'logit' else sm.OLS(d[y].astype(float), X)
    try:
        res = model.fit(disp=0, maxiter=300, cov_type='cluster', cov_kwds={'groups': g}) if kind == 'logit' \
              else model.fit(cov_type='cluster', cov_kwds={'groups': g})
    except Exception as e:
        print('   fit failed:', e); return None, None
    ci = res.conf_int()
    tab = pd.DataFrame({'coef_perSD': res.params, 'se_cluster': res.bse, 'p': res.pvalues,
                        'ci_low': ci[0], 'ci_high': ci[1]})
    if kind == 'logit':
        tab['OR_perSD'] = np.exp(tab.coef_perSD); tab['OR_low'] = np.exp(tab.ci_low); tab['OR_high'] = np.exp(tab.ci_high)
        fit = dict(n=int(res.nobs), n_clusters=int(len(np.unique(g))), pseudo_R2=res.prsquared, llr_p=res.llr_pvalue)
    else:
        fit = dict(n=int(res.nobs), n_clusters=int(len(np.unique(g))), R2=res.rsquared)
    return tab.drop(index='const'), fit

def cohens_d(a, b):
    a, b = np.asarray(a, float), np.asarray(b, float)
    sp = np.sqrt(((len(a) - 1) * a.var(ddof=1) + (len(b) - 1) * b.var(ddof=1)) / (len(a) + len(b) - 2))
    return (a.mean() - b.mean()) / sp if sp > 0 else np.nan

def cluster_boot_diff(df, col, g1, g2, n_boot=None, seed=SEED):
    n_boot = n_boot or CONFIG['N_BOOT']; rng = np.random.default_rng(seed)
    cids = df.cid.unique(); by = {c: d for c, d in df.groupby('cid')}
    diffs = []
    for _ in range(n_boot):
        s = pd.concat([by[c] for c in rng.choice(cids, len(cids), replace=True)])
        a, b = s.loc[s.label == g1, col], s.loc[s.label == g2, col]
        if len(a) and len(b): diffs.append(a.mean() - b.mean())
    return np.percentile(diffs, [2.5, 97.5])

def verdict(p_holm, sign, expected):
    if p_holm is None or np.isnan(p_holm): return 'not testable'
    if p_holm >= 0.05: return 'not supported'
    return 'supported' if np.sign(sign) == np.sign(expected) else 'reversed (significant, opposite sign)'

def meta_random_effects(est, se, se_floor=1e-3):
    # DerSimonian-Laird tau^2 with Hartung-Knapp-Sidik-Jonkman CI / p (t with k-1 df; HKSJ variance never
    # smaller than the DL variance). Appropriate for a small number of studies (here: datasets).
    # se_floor: a bootstrap SE of exactly 0 (two models with identical predictions) would get infinite weight.
    est, se = np.asarray(est, float), np.asarray(se, float)
    ok = np.isfinite(est) & np.isfinite(se); est, se = est[ok], np.maximum(se[ok], se_floor); k = len(est)
    if k < 2: return None
    w = 1 / se ** 2; fe = (w * est).sum() / w.sum(); Q = (w * (est - fe) ** 2).sum()
    tau2 = max(0.0, (Q - (k - 1)) / (w.sum() - (w ** 2).sum() / w.sum()))
    wr = 1 / (se ** 2 + tau2); mu = (wr * est).sum() / wr.sum()
    var_dl = 1 / wr.sum(); var_hk = (wr * (est - mu) ** 2).sum() / ((k - 1) * wr.sum())
    se_mu = np.sqrt(max(var_dl, var_hk)); tq = stats.t.ppf(0.975, k - 1)
    return dict(k=k, pooled=mu, se=se_mu, ci_low=mu - tq * se_mu, ci_high=mu + tq * se_mu,
                p=2 * stats.t.sf(abs(mu / se_mu), k - 1), tau2=tau2, I2=max(0.0, (Q - (k - 1)) / Q) if Q > 0 else 0.0, Q=Q)

In [ ]:
CACHE, SNAPS, RESULTS = {}, {}, {}

EVENTS, WINDOW = {}, {}    # WINDOW[name] = (m, k): width = m * stride, label step k
def _mk(name, m=None, k=None):
    m0, k0 = WINDOW.get(name, (1, 1))
    return (m0 if m is None else m), (k0 if k is None else k)

def get_events(name):
    if name not in EVENTS:
        ev = load_events(name, DATASETS[name]); EVENTS[name] = ev
        print(f'  {name}: {len(ev):,} events, {pd.unique(ev[["src", "dst"]].values.ravel()).size:,} nodes, '
              f'{(ev.ts.max() - ev.ts.min()) / 86400:.0f} days')
    return EVENTS[name]

def get_snapshots(name, m=None):
    m, _ = _mk(name, m)
    if (name, m) not in SNAPS:
        s = DATASETS[name]['stride_days']
        SNAPS[(name, m)] = build_snapshots(get_events(name), m * s, s)
    return SNAPS[(name, m)]

def get_partition_stats(name, algo, seed, m=None):
    m, _ = _mk(name, m)
    key = (name, m, algo, seed)
    if key not in CACHE:
        snaps = get_snapshots(name, m); t = time.time()
        parts = [detect_communities(s['G'], algo, seed, CONFIG['MIN_COMM_SIZE']) for s in snaps]
        CACHE[key] = (parts, snapshot_stats(snaps, parts))
        print(f'  {name} [width {m}x{DATASETS[name]["stride_days"]}d, {algo}, seed={seed}]: '
              f'{len(snaps)} snapshots, {sum(map(len, parts))} community-snapshots ({time.time()-t:.0f}s)')
    return CACHE[key]

def make_table(name, algo=None, seed=None, theta=None, label_thr=None, m=None, k=None, death_as_shrink=None):
    algo = algo or CONFIG['CD_ALGO']; seed = CONFIG['CD_SEED'] if seed is None else seed
    theta = CONFIG['JACCARD_THETA'] if theta is None else theta
    label_thr = CONFIG['LABEL_THRESHOLD'] if label_thr is None else label_thr
    m, k = _mk(name, m, k)
    parts, stats = get_partition_stats(name, algo, seed, m)
    ids = track_communities(parts, theta)
    das = CONFIG['DEATH_AS_SHRINK'] if death_as_shrink is None else death_as_shrink
    return build_table(stats, ids, len(parts), label_thr, das, h=k)

In [ ]:
# windows chosen by the v3 yield rule (table_window_selection.csv, chosen & meets_rule)
DIAG_WINDOWS = {'email-Eu-core': (1, 1), 'mathoverflow': (2, 2), 'reddit-hyperlinks': (1, 1), 'superuser': (4, 4)}
NOISE_SEEDS = [0, 1, 2, 3, 4]
if SYNTHETIC_SMOKE_TEST:
    DIAG_WINDOWS = {'synthA': (1, 1), 'synthB': (1, 1)}; NOISE_SEEDS = [0, 1]
WINDOW.update(DIAG_WINDOWS)
print('Diagnostics on:', DIAG_WINDOWS)

In [ ]:
def matched_sizes(name):
    """For every seed-42 community (t, j): its size under each alternative Louvain seed, matched by best Jaccard
    (>= JACCARD_THETA, else NaN). Returns dict (t, cid) -> array of sizes [seed42, seeds...]."""
    m, k = WINDOW[name]; snaps = get_snapshots(name, m)
    parts0, _ = get_partition_stats(name, CONFIG['CD_ALGO'], CONFIG['CD_SEED'], m)
    ids = track_communities(parts0, CONFIG['JACCARD_THETA'])
    alt = []
    for s in NOISE_SEEDS:
        t0 = time.time(); alt.append([detect_communities(S['G'], CONFIG['CD_ALGO'], s, CONFIG['MIN_COMM_SIZE']) for S in snaps])
        print(f'   {name} seed {s}: {time.time() - t0:.0f}s')
    out = {}
    for t, comms in enumerate(parts0):
        maps = [{u: i for i, c in enumerate(P[t]) for u in c} for P in alt]
        for j, C in enumerate(comms):
            row = [len(C)]
            for P, mp in zip(alt, maps):
                ov = Counter(mp[u] for u in C if u in mp)
                best = max(((o / (len(P[t][i]) + len(C) - o), len(P[t][i])) for i, o in ov.items()), default=(0, np.nan))
                row.append(best[1] if best[0] >= CONFIG['JACCARD_THETA'] else np.nan)
            out[(t, ids[t][j])] = np.array(row, float)
    return out

def reversal_stats(d, x='rel_d_size', y='rel_change_next'):
    d = d[[x, y, 'cid']].replace([np.inf, -np.inf], np.nan).dropna()
    if len(d) < 20: return dict(n=len(d))
    X = sm.add_constant(d[[x]]); res = sm.OLS(d[y], X).fit(cov_type='cluster', cov_kwds={'groups': pd.factorize(d.cid)[0]})
    rho, p_rho = stats.spearmanr(d[x], d[y])
    return dict(n=len(d), slope=res.params[x], slope_lo=res.conf_int().loc[x, 0], slope_hi=res.conf_int().loc[x, 1],
                slope_p=res.pvalues[x], spearman=rho, spearman_p=p_rho)

def kappa(a, b):
    a, b = np.asarray(a), np.asarray(b); po = (a == b).mean()
    pe = sum((a == c).mean() * (b == c).mean() for c in CLASSES)
    return (po - pe) / (1 - pe) if pe < 1 else np.nan

DIAG = []
for name in DIAG_WINDOWS:
    print(f'\n=== {name} ==='); t_start = time.time()
    m, k = WINDOW[name]; n_snap = len(get_snapshots(name, m))
    tab = make_table(name); inst = instances_from(tab)
    R = dict(dataset=name, m=m, k=k, N=len(inst))
    # ---- D1: reversal on the primary (seed-42) sizes
    R.update({f'D1_{kk}': v for kk, v in reversal_stats(inst).items()})
    # ---- D2: label reliability across Louvain seeds
    MS = matched_sizes(name)
    def sizes_at(cid, t): return MS.get((t, cid), np.full(len(NOISE_SEEDS) + 1, np.nan))
    S_t = np.vstack([sizes_at(c, t) for c, t in zip(inst.cid, inst.t)])
    S_next = np.vstack([sizes_at(c, t + k) for c, t in zip(inst.cid, inst.t)])
    S_prev = np.vstack([sizes_at(c, t - 1) for c, t in zip(inst.cid, inst.t)])
    rel_sd = np.nanstd(S_t, 1) / np.nanmean(S_t, 1)
    R['D2_median_rel_sd_size_across_seeds'] = float(np.nanmedian(rel_sd))
    R['D2_median_abs_next_change'] = float(inst.rel_change_next.abs().median())
    agree, kap = [], []
    for s in range(1, S_t.shape[1]):
        ok = ~np.isnan(S_t[:, s]) & ~np.isnan(S_next[:, s])
        lab_s = np.array([label_from_change((b - a) / a, CONFIG['LABEL_THRESHOLD']) for a, b in zip(S_t[ok, s], S_next[ok, s])])
        agree.append((lab_s == inst.label.values[ok]).mean()); kap.append(kappa(lab_s, inst.label.values[ok]))
        R[f'D2_matched_share_seed{NOISE_SEEDS[s - 1]}'] = ok.mean()
    R['D2_label_agreement_mean'] = float(np.mean(agree)); R['D2_kappa_mean'] = float(np.mean(kap))
    R['D2_kappa_min'] = float(np.min(kap)); R['D2_kappa_max'] = float(np.max(kap))
    # ---- D3: reversal on seed-averaged sizes (less noisy measurement)
    avg = lambda S: np.nanmean(S, 1)
    d3 = inst[['cid']].copy()
    d3['rel_d_size'] = (avg(S_t) - avg(S_prev)) / avg(S_prev); d3['rel_change_next'] = (avg(S_next) - avg(S_t)) / avg(S_t)
    R.update({f'D3_{kk}': v for kk, v in reversal_stats(d3).items()})
    lab_avg = d3.rel_change_next.apply(lambda r: label_from_change(r, CONFIG['LABEL_THRESHOLD']) if np.isfinite(r) else None)
    pers_avg = d3.rel_d_size.apply(lambda r: label_from_change(r, CONFIG['LABEL_THRESHOLD']) if np.isfinite(r) else None)
    ok = lab_avg.notna() & pers_avg.notna()
    R['D3_persistence_macro_f1_avg_labels'] = macro_f1(lab_avg[ok], pers_avg[ok])
    R['D1_persistence_macro_f1'] = macro_f1(inst.label, persistence_baseline(inst, CONFIG['LABEL_THRESHOLD']))
    # ---- D4: two-feature mean-reversion model vs combined RF (purged rolling origin, as in v3)
    F = make_folds(inst, n_snap, m, k)
    if F:
        tr0 = F[0][2]; _, prm, _ = tune_rf(tr0[COMBINED], tr0.label, tr0.cid)
        seeds = [SEED + i for i in range(CONFIG['N_RF_SEEDS'])]
        y = np.concatenate([te.label.values for *_, te in F]); g = np.concatenate([te.cid.values for *_, te in F])
        rf = np.concatenate([rf_seed_ensemble(tr, te, COMBINED, prm, seeds)[0] for _, _, tr, te in F])
        REV = ['rel_d_size', 'log_size']
        lr = np.concatenate([make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, class_weight='balanced'))
                             .fit(tr[REV], tr.label).predict(te[REV]) for _, _, tr, te in F])
        d, ci, p = paired_cluster_boot(y, rf, lr, g)
        R.update(D4_N_eval=len(y), D4_rf_combined_macro_f1=macro_f1(y, rf), D4_reversion_logit_macro_f1=macro_f1(y, lr),
                 D4_rf_minus_reversion=d, D4_ci_low=ci[0], D4_ci_high=ci[1], D4_p_boot=p)
    R['runtime_s'] = time.time() - t_start; DIAG.append(R)
    pd.DataFrame(DIAG).to_csv(os.path.join(RESULTS_DIR, 'table_diagnostics.csv'), index=False)   # checkpoint
DIAG = pd.DataFrame(DIAG); DIAG.to_csv(os.path.join(RESULTS_DIR, 'table_diagnostics.csv'), index=False)
display(DIAG.T)

meta_d4 = meta_random_effects(DIAG.D4_rf_minus_reversion, (DIAG.D4_ci_high - DIAG.D4_ci_low) / (2 * 1.96)) if 'D4_rf_minus_reversion' in DIAG else None
if meta_d4:
    print('Pooled RF(combined) - reversion logit:', {k: round(v, 4) for k, v in meta_d4.items()})
    pd.DataFrame([meta_d4]).to_csv(os.path.join(RESULTS_DIR, 'table_meta_D4.csv'), index=False)

In [ ]:
archive = shutil.make_archive(os.path.join(BASE_DIR, 'results_diagnostics'), 'zip', RESULTS_DIR)
print('Wrote', archive)
if IN_COLAB:
    if SAVE_TO_DRIVE:
        from google.colab import drive; drive.mount('/content/drive')
        shutil.copy(archive, '/content/drive/MyDrive/'); print('Copied to Drive')
    from google.colab import files; files.download(archive)